# Adversarial Debate — LangGraph

**Pattern:** proposer → critic → judge (optionally looped for multiple rounds)

```
START → proposer → critic ─┬─→ proposer (round 2+)
                           └─→ judge → END
```

State accumulates the full debate transcript. The routing function controls how many rounds to run.

In [ ]:
import os
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)
print("✓ Ready")

In [ ]:
class DebateState(TypedDict):
    claim: str; proposal: Optional[str]; critique: Optional[str]; verdict: Optional[str]; round: int

def proposer_node(s):
    r = llm.invoke([SystemMessage(content="STRONGEST case FOR the claim. 3 arguments, 150 words."),
                    HumanMessage(content=f"Claim: {s['claim']}\n\nArgue FOR:")])
    return {"proposal": r.content}

def critic_node(s):
    r = llm.invoke([SystemMessage(content="Argue AGAINST. Find weaknesses. 150 words."),
                    HumanMessage(content=f"Claim: {s['claim']}\n\nFOR:\n{s['proposal']}\n\nArgue AGAINST:")])
    return {"critique": r.content, "round": s["round"] + 1}

def judge_node(s):
    r = llm.invoke([SystemMessage(content="Score each side 1-10, name stronger side, conclusion."),
                    HumanMessage(content=f"Claim: {s['claim']}\n\nFOR:\n{s['proposal']}\n\nAGAINST:\n{s['critique']}\n\nVerdict:")])
    return {"verdict": r.content}

print("3 nodes defined")

In [ ]:
MAX_ROUNDS = 1  # change to 2 for a second debate round

builder = StateGraph(DebateState)
builder.add_node("proposer", proposer_node)
builder.add_node("critic", critic_node)
builder.add_node("judge", judge_node)
builder.add_edge(START, "proposer")
builder.add_edge("proposer", "critic")
builder.add_conditional_edges("critic",
    lambda s: "proposer" if s["round"] < MAX_ROUNDS else "judge",
    {"proposer": "proposer", "judge": "judge"})
builder.add_edge("judge", END)
graph = builder.compile()

try:
    from IPython.display import Image, display; display(Image(graph.get_graph().draw_mermaid_png()))
except: print(graph.get_graph().draw_mermaid())

In [ ]:
claim = "Tokyo is the best travel destination for a one-week trip"
result = graph.invoke({"claim": claim, "proposal": None, "critique": None, "verdict": None, "round": 0})

print(f"Claim: {claim}")
print(f"\n--- FOR ---\n{result['proposal']}")
print(f"\n--- AGAINST ---\n{result['critique']}")
print(f"\n--- VERDICT ---\n{result['verdict']}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `round` in state | Enables multi-round debates — loop counter lives in state |
| Conditional edge after critic | `MAX_ROUNDS` controls loop termination |
| Judge receives both from state | State accumulates full debate — judge sees everything |

**LangGraph advantage for debate:** The loop edge (`critic → proposer`) is explicit and bounded — unlike LangChain where you'd manually call in a Python loop.

### Next: [CrewAI Debate →](../CrewAI/debate.ipynb)